> 所谓xx并行，就是xx拆到不同的卡上，
- 数据并行，就是数据拆到不同的卡上（不同卡处理不同的数据）；
- 张量并行，就是 tensor 拆分到不同的卡上；

In [1]:
from IPython.display import Image

- MoE 训练用 TP 还是 EP？
    - 总参数与激活量参数计算
- https://huggingface.co/blog/moe
- https://arxiv.org/pdf/2401.06066
    - DeepSeekMoE
        - Experts: routed expert & shared expert

In [2]:
Image(url='./figs/deepseek-moe.png', width=400)

- 2/N => 4/2N => (1+3)/2N

## MoE

- MoE transformer block
    - shared part：自注意力层；
        - 这些是所有Token都必须经过的层。在Transformer中，这主要是自注意力（Self-Attention）层和一些层归一化（LayerNorm）等。这些部分的参数量相对较小。
        - 对于共享部分（如Attention层），我们采用经典的数据并行（DP）。这意味着，Attention层的完整参数被复制到了每一个GPU上。
    - moe part：gating network/router + experts
        - 这特指MoE层中的多个专家网络（Experts）。这部分的参数量是整个模型中最大的。一个专家本身就是一个FFN（前馈网络）。
        - 对于专家部分（Experts），我们采用专家并行（EP）。这意味着，所有专家的总参数被分散（sharded）到了不同的GPU上，每个GPU只持有总专家库的一个子集。

### 参数量计算

### EP

- 2个gpu `[GPU0, GPU1]`，4个experts `[E0, E1, E2, E3]`，6个input tokens `[T0, T1, T2, T3, T4, T5]`
- DP & EP
    - GPU-0 存储着:一份完整的Attention层参数（副本），一份完整的Router参数（副本，因为它也算共享部分），专家 E0 和 E1 的参数（独有）
    - GPU-1 存储着:一份完整的Attention层参数（副本），一份完整的Router参数（副本），专家 E2 和 E3 的参数（独有）
- 第一步： 计算Attention层 (纯数据并行)
    - GPU-0 使用它本地的Attention层副本，独立计算出 `[T1, T2, T3]` 的Attention输出。
    - GPU-1 使用它本地的Attention层副本，独立计算出 `[T4, T5, T6]` 的Attention输出。
    - 到此为止，一切都和标准的数据并行完全一样。
- 第 2 步: 进入MoE层 (从DP切换到EP模式)
    - 路由计算:
        - GPU-0 上的Router副本计算出 `[T1, T2, T3]` 的目标专家。
            - GPU-0 持有 `[T1, T2, T3]`，并且知道它们的目的地分别是 `[E2, E0, E2]`。
        - GPU-1 上的Router副本计算出 `[T4, T5, T6]` 的目标专家。
            - GPU-1 持有 `[T4, T5, T6]`，并且知道它们的目的地分别是 `[E3, E0, E1]`。
    - All-to-All 通信:
        - Token们根据目标专家的位置，被发送到对应的GPU。
        - 例如，T1（在GPU-0上）需要去E2（在GPU-1上），所以T1的向量数据通过网络被发送到GPU-1。
            - GPU-0 收到了 来自 GPU-1 的 `[T5, T6]`。它现在持有的Token是 `[T2, T5, T6]`。
            - GPU-1 收到了 来自 GPU-0 的 `[T1, T3]`。它现在持有的Token是 `[T4, T1, T3]`。
        - Token不再按照原始批次分组，而是按照它们需要访问的专家所在的位置重新分组了。
    - 专家计算:
        - GPU-0 使用它独有的专家E0和E1进行计算。
        - GPU-1 使用它独有的专家E2和E3进行计算。
    - 第二次 All-to-All 通信:
        - 计算结果被送回Token原始的GPU。
    - 聚合:每个GPU将收到的结果聚合，完成MoE层的计算。